# 实验二 ConvLSTM 的算法应用与改进

本 Notebook 严格对应实验报告样板：环境、数据、基线配置、参数量手算核对、6 组控制变量实验、最优组合三次复测、预测帧九宫格及报告可用汇总。全部实验使用 CPU；实验二输出统一保存到 `lab02_outputs/`，避免覆盖实验一文件。

## 1 实验环境与随机种子

In [ ]:
# 本机环境先加载 sklearn，可避免其与 PyTorch 的 OpenMP 运行库加载冲突。
import sklearn
import csv
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import Markdown, display
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
BEST_SEEDS = (42, 43, 44)
DEVICE = torch.device('cpu')
ROOT = Path.cwd().resolve()
if ROOT.name == 'lab02_convlstm':
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / 'lab02_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)
print('操作系统:', platform.platform())
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)
print('Matplotlib:', matplotlib.__version__)
print('CUDA 可用:', torch.cuda.is_available())
print('本实验设备:', DEVICE)

## 2 数据合成

按指南正文生成 2000 条训练序列和 200 条独立测试序列，共 2200 条。每条包含 10 帧 32×32 图像，小球半径为 2，初速度绝对值为 0.8～1.6 像素/帧。

In [ ]:
def make_sequences(n_seq=2200, total_frames=10, size=32, radius=2, seed=42):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]
    sequences = np.zeros((n_seq, total_frames, size, size), dtype=np.uint8)

    for sequence_index in range(n_seq):
        x, y = rng.uniform(radius + 1, size - radius - 2, size=2)
        direction = rng.choice([-1.0, 1.0], size=2)
        vx, vy = direction * rng.uniform(0.8, 1.6, size=2)

        for frame_index in range(total_frames):
            x += vx
            y += vy
            if x < radius:
                x = 2 * radius - x
                vx = -vx
            elif x > size - 1 - radius:
                x = 2 * (size - 1 - radius) - x
                vx = -vx
            if y < radius:
                y = 2 * radius - y
                vy = -vy
            elif y > size - 1 - radius:
                y = 2 * (size - 1 - radius) - y
                vy = -vy
            sequences[sequence_index, frame_index] = (
                (xx - x) ** 2 + (yy - y) ** 2 <= radius ** 2
            )
    return sequences

all_sequences = torch.from_numpy(make_sequences(seed=SEED)).float().unsqueeze(2)
train_sequences = all_sequences[:2000]
test_sequences = all_sequences[2000:]
print('全部数据形状:', tuple(all_sequences.shape))
print('训练序列:', tuple(train_sequences.shape))
print('测试序列:', tuple(test_sequences.shape))
print('训练/测试是否分离:', train_sequences.data_ptr() != test_sequences.data_ptr())

## 3 ConvLSTM 与全连接 LSTM 模型

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.gates = nn.Conv2d(
            input_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size,
            padding=kernel_size // 2,
        )

    def forward(self, inputs, state):
        hidden, cell = state
        gates = self.gates(torch.cat([inputs, hidden], dim=1))
        input_gate, forget_gate, candidate, output_gate = gates.chunk(4, dim=1)
        input_gate = torch.sigmoid(input_gate)
        forget_gate = torch.sigmoid(forget_gate)
        output_gate = torch.sigmoid(output_gate)
        cell_new = forget_gate * cell + input_gate * torch.tanh(candidate)
        hidden_new = output_gate * torch.tanh(cell_new)
        return hidden_new, cell_new


class ConvLSTM(nn.Module):
    def __init__(self, input_channels=1, hidden_channels=32, kernel_size=3, layers=1):
        super().__init__()
        channels = [input_channels] + [hidden_channels] * layers
        self.cells = nn.ModuleList([
            ConvLSTMCell(channels[index], channels[index + 1], kernel_size)
            for index in range(layers)
        ])
        self.output_conv = nn.Conv2d(hidden_channels, 1, 3, padding=1)

    def forward(self, inputs):
        batch_size, time_steps, _, height, width = inputs.shape
        states = [
            (
                inputs.new_zeros(batch_size, cell.hidden_channels, height, width),
                inputs.new_zeros(batch_size, cell.hidden_channels, height, width),
            )
            for cell in self.cells
        ]
        for time_index in range(time_steps):
            layer_inputs = inputs[:, time_index]
            for layer_index, cell in enumerate(self.cells):
                states[layer_index] = cell(layer_inputs, states[layer_index])
                layer_inputs = states[layer_index][0]
        return self.output_conv(states[-1][0])


class FlattenLSTM(nn.Module):
    def __init__(self, hidden_size=256, image_size=32):
        super().__init__()
        self.image_size = image_size
        pixels = image_size * image_size
        self.lstm = nn.LSTM(pixels, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, pixels)

    def forward(self, inputs):
        batch_size, time_steps = inputs.shape[:2]
        flattened = inputs.reshape(batch_size, time_steps, -1)
        sequence_output, _ = self.lstm(flattened)
        prediction = self.output(sequence_output[:, -1])
        return prediction.reshape(batch_size, 1, self.image_size, self.image_size)

## 4 参数量手算与代码核对

基线细胞：`4 × (3×3×1×32 + 3×3×32×32 + 32)`；输出卷积：`3×3×32×1 + 1`。本节同时给出报告思考题所需的 `C_h=64` 和全连接 LSTM 参数量。

In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())

baseline_cell_hand = 4 * (3 * 3 * 1 * 32 + 3 * 3 * 32 * 32 + 32)
baseline_output_hand = 3 * 3 * 32 * 1 + 1
baseline_total_hand = baseline_cell_hand + baseline_output_hand
hidden64_cell_hand = 4 * (3 * 3 * 1 * 64 + 3 * 3 * 64 * 64 + 64)
hidden64_output_hand = 3 * 3 * 64 * 1 + 1

baseline_model_for_count = ConvLSTM()
hidden64_model_for_count = ConvLSTM(hidden_channels=64)
flatten_model_for_count = FlattenLSTM()

print('基线 ConvLSTM 细胞手算:', baseline_cell_hand)
print('基线输出卷积手算:', baseline_output_hand)
print('基线总参数量手算:', baseline_total_hand)
print('基线总参数量代码核对:', count_parameters(baseline_model_for_count))
print('C_h=64 单细胞参数量:', hidden64_cell_hand)
print('C_h=64 模型总参数量:', count_parameters(hidden64_model_for_count))
print('全连接 LSTM 总参数量:', count_parameters(flatten_model_for_count))
print('C_h=64 细胞/基线细胞倍数:', hidden64_cell_hand / baseline_cell_hand)
assert baseline_total_hand == count_parameters(baseline_model_for_count)

## 5 实验配置、训练、评估与绘图

基线为 1 层、`C_h=32`、`K=3`、看前 4 帧、MSE、Adam(0.001)、批量 64、5 epoch。实验 1～6 均从基线配置替换指南指定变量。

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    key: str
    group: str
    name: str
    change: str
    model_type: str = 'convlstm'
    layers: int = 1
    hidden_channels: int = 32
    kernel_size: int = 3
    input_frames: int = 4
    loss_name: str = 'mse'
    learning_rate: float = 0.001
    batch_size: int = 64
    epochs: int = 5

BASELINE = ExperimentConfig('baseline', '0', '基线 Baseline', '1层，C_h=32，K=3，看4帧，MSE')
EXPERIMENTS = [
    BASELINE,
    replace(BASELINE, key='exp1', group='1', name='加深层数', change='1层 → 2层', layers=2),
    replace(BASELINE, key='exp2', group='2', name='隐藏通道', change='C_h: 32 → 64', hidden_channels=64),
    replace(BASELINE, key='exp3', group='3', name='卷积核', change='K: 3 → 5', kernel_size=5),
    replace(BASELINE, key='exp4', group='4', name='输入帧数', change='看4帧 → 看8帧', input_frames=8),
    replace(BASELINE, key='exp5', group='5', name='结构对照', change='ConvLSTM → 全连接LSTM', model_type='flatten_lstm'),
    replace(BASELINE, key='exp6', group='6', name='损失函数', change='MSE → L1', loss_name='l1'),
]

def build_model(config):
    if config.model_type == 'flatten_lstm':
        return FlattenLSTM()
    return ConvLSTM(
        hidden_channels=config.hidden_channels,
        kernel_size=config.kernel_size,
        layers=config.layers,
    )

def make_loaders(config, seed):
    train_x = train_sequences[:, :config.input_frames]
    train_y = train_sequences[:, config.input_frames]
    test_x = test_sequences[:, :config.input_frames]
    test_y = test_sequences[:, config.input_frames]
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        TensorDataset(train_x, train_y),
        batch_size=config.batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    test_loader = DataLoader(
        TensorDataset(test_x, test_y),
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, test_loader

def evaluate(model, data_loader):
    model.eval()
    squared_error = 0.0
    absolute_error = 0.0
    element_count = 0
    with torch.no_grad():
        for inputs, targets in data_loader:
            predictions = model(inputs.to(DEVICE))
            targets = targets.to(DEVICE)
            squared_error += (predictions - targets).pow(2).sum().item()
            absolute_error += (predictions - targets).abs().sum().item()
            element_count += targets.numel()
    return squared_error / element_count, absolute_error / element_count

def train_experiment(config, seed=42):
    set_seed(seed)
    model = build_model(config).to(DEVICE)
    loss_function = nn.MSELoss() if config.loss_name == 'mse' else nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    train_loader, test_loader = make_loaders(config, seed)
    history = {'train_loss': [], 'test_mse': [], 'test_mae': []}
    start_time = time.perf_counter()

    for epoch in range(config.epochs):
        model.train()
        epoch_loss = 0.0
        sample_count = 0
        for inputs, targets in train_loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = loss_function(predictions, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * inputs.shape[0]
            sample_count += inputs.shape[0]
        train_loss = epoch_loss / sample_count
        test_mse, test_mae = evaluate(model, test_loader)
        history['train_loss'].append(train_loss)
        history['test_mse'].append(test_mse)
        history['test_mae'].append(test_mae)
        print(
            f'{config.key} epoch {epoch + 1}/{config.epochs} '
            f'train_loss={train_loss:.6f} test_MSE={test_mse:.6f} test_MAE={test_mae:.6f}'
        )
    elapsed = time.perf_counter() - start_time
    return model, history, elapsed

def save_history(config, history):
    path = OUTPUT_DIR / f'history_{config.key}.csv'
    with path.open('w', newline='', encoding='utf-8-sig') as file:
        writer = csv.writer(file)
        writer.writerow(['epoch', 'train_loss', 'test_mse', 'test_mae'])
        for index in range(config.epochs):
            writer.writerow([
                index + 1, history['train_loss'][index],
                history['test_mse'][index], history['test_mae'][index],
            ])

def save_curves(config, history, filename=None):
    epochs = range(1, config.epochs + 1)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history['train_loss'], marker='o', label='训练损失')
    axes[0].set_title('训练损失曲线')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.25)
    axes[0].legend()
    axes[1].plot(epochs, history['test_mse'], marker='o', label='测试 MSE')
    axes[1].plot(epochs, history['test_mae'], marker='s', label='测试 MAE')
    axes[1].set_title('测试误差曲线')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Error')
    axes[1].grid(alpha=0.25)
    axes[1].legend()
    figure.suptitle(f'组 {config.group}：{config.name}')
    figure.tight_layout()
    output_name = filename or f'result_{config.key}.png'
    figure.savefig(OUTPUT_DIR / output_name, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(figure)

## 6 基线与六组对照实验

本单元会真实训练 7 个模型并保存 `result_baseline.png`、`result_exp1.png`～`result_exp6.png`、逐轮 CSV 和汇总 `results.csv`。

In [ ]:
experiment_rows = []
retained_models = {}
all_histories = {}

for config in EXPERIMENTS:
    print(f'\n===== 组 {config.group} {config.name} =====')
    model, history, elapsed = train_experiment(config, SEED)
    save_history(config, history)
    save_curves(config, history)
    final_mse = history['test_mse'][-1]
    final_mae = history['test_mae'][-1]
    row = {
        'group': config.group,
        'key': config.key,
        'name': config.name,
        'change': config.change,
        'test_mse': final_mse,
        'test_mae': final_mae,
        'training_seconds': elapsed,
        'parameters': count_parameters(model),
        'seed': SEED,
    }
    experiment_rows.append(row)
    all_histories[config.key] = history
    if config.key in ('baseline', 'exp6'):
        retained_models[config.key] = model
    print(
        f'最终：MSE={final_mse:.6f}, MAE={final_mae:.6f}, '
        f'耗时={elapsed:.1f}s, 参数量={count_parameters(model):,}'
    )

with (OUTPUT_DIR / 'results.csv').open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=list(experiment_rows[0].keys()))
    writer.writeheader()
    writer.writerows(experiment_rows)

## 7 最优组合选择与三次复测

从层数、隐藏通道、卷积核、输入帧数和损失函数五项可组合改进中，按单项测试 MSE 下降幅度选择效果最明显的两项；全连接 LSTM 属于替代结构，不与 ConvLSTM 组合。三次复测使用预先固定的种子 42、43、44，并完整记录。

In [ ]:
config_by_key = {config.key: config for config in EXPERIMENTS}
row_by_key = {row['key']: row for row in experiment_rows}
combinable_keys = ('exp1', 'exp2', 'exp3', 'exp4', 'exp6')
effective_keys = [
    key for key in combinable_keys
    if row_by_key[key]['test_mse'] < baseline_mse
]
selected_keys = sorted(effective_keys, key=lambda key: row_by_key[key]['test_mse'])[:2]
best_config = replace(BASELINE, key='best', group='7', name='最优组合')
for key in selected_keys:
    source = config_by_key[key]
    if key == 'exp1':
        best_config = replace(best_config, layers=source.layers)
    elif key == 'exp2':
        best_config = replace(best_config, hidden_channels=source.hidden_channels)
    elif key == 'exp3':
        best_config = replace(best_config, kernel_size=source.kernel_size)
    elif key == 'exp4':
        best_config = replace(best_config, input_frames=source.input_frames)
    elif key == 'exp6':
        best_config = replace(best_config, loss_name=source.loss_name)
best_config = replace(
    best_config,
    change=' + '.join(config_by_key[key].change for key in selected_keys),
)
print('单项有效改进:', effective_keys)
print('最优组合采用:', selected_keys)
print('最优组合配置:', asdict(best_config))

In [ ]:
best_rows = []
best_histories = []
best_model_for_prediction = None
for run_number, seed in enumerate(BEST_SEEDS, start=1):
    print(f'\n===== 最优组合第 {run_number} 次，seed={seed} =====')
    model, history, elapsed = train_experiment(best_config, seed)
    best_histories.append(history)
    best_rows.append({
        'run': run_number,
        'seed': seed,
        'test_mse': history['test_mse'][-1],
        'test_mae': history['test_mae'][-1],
        'training_seconds': elapsed,
    })
    save_history(replace(best_config, key=f'best_run{run_number}'), history)
    if run_number == 1:
        best_model_for_prediction = model

with (OUTPUT_DIR / 'best_runs_results.csv').open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=list(best_rows[0].keys()))
    writer.writeheader()
    writer.writerows(best_rows)

mean_mse = float(np.mean([row['test_mse'] for row in best_rows]))
mean_mae = float(np.mean([row['test_mae'] for row in best_rows]))
mean_time = float(np.mean([row['training_seconds'] for row in best_rows]))
std_mse = float(np.std([row['test_mse'] for row in best_rows]))
print(f'最优组合平均：MSE={mean_mse:.6f} ± {std_mse:.6f}, MAE={mean_mae:.6f}, 耗时={mean_time:.1f}s')

mean_history = {
    metric: np.mean([history[metric] for history in best_histories], axis=0).tolist()
    for metric in ('train_loss', 'test_mse', 'test_mae')
}
save_curves(best_config, mean_history, filename='result_best.png')

## 8 预测帧九宫格与 L1/MSE 视觉对照

In [ ]:
def predict_samples(model, config, indices=(0, 1, 2)):
    inputs = test_sequences[list(indices), :config.input_frames].to(DEVICE)
    targets = test_sequences[list(indices), config.input_frames].to(DEVICE)
    model.eval()
    with torch.no_grad():
        predictions = model(inputs)
    sample_mse = (predictions - targets).pow(2).flatten(1).mean(dim=1)
    return inputs.cpu(), targets.cpu(), predictions.cpu(), sample_mse.cpu()

sample_indices = (0, 37, 101)
inputs, targets, predictions, sample_mse = predict_samples(
    best_model_for_prediction, best_config, sample_indices
)
figure, axes = plt.subplots(3, 3, figsize=(9, 9))
for row_index, sample_index in enumerate(sample_indices):
    images = [
        inputs[row_index, -1, 0], targets[row_index, 0], predictions[row_index, 0].clamp(0, 1)
    ]
    titles = [
        f'样本 {sample_index} 最后一帧输入',
        '真实下一帧',
        f'预测帧\nMSE={sample_mse[row_index]:.6f}',
    ]
    for column_index, (image, title) in enumerate(zip(images, titles)):
        axes[row_index, column_index].imshow(image, cmap='gray', vmin=0, vmax=1)
        axes[row_index, column_index].set_title(title)
        axes[row_index, column_index].axis('off')
figure.suptitle('输入帧 | 真实下一帧 | ConvLSTM 预测帧')
figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'result_pred.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(figure)

In [ ]:
baseline_inputs, baseline_targets, baseline_predictions, _ = predict_samples(
    retained_models['baseline'], BASELINE, sample_indices
)
_, _, l1_predictions, _ = predict_samples(
    retained_models['exp6'], config_by_key['exp6'], sample_indices
)
figure, axes = plt.subplots(3, 4, figsize=(12, 9))
for row_index, sample_index in enumerate(sample_indices):
    images = [
        baseline_inputs[row_index, -1, 0], baseline_targets[row_index, 0],
        baseline_predictions[row_index, 0].clamp(0, 1), l1_predictions[row_index, 0].clamp(0, 1),
    ]
    titles = [f'样本 {sample_index} 输入', '真实帧', 'MSE训练预测', 'L1训练预测']
    for column_index, (image, title) in enumerate(zip(images, titles)):
        axes[row_index, column_index].imshow(image, cmap='gray', vmin=0, vmax=1)
        axes[row_index, column_index].set_title(title)
        axes[row_index, column_index].axis('off')
figure.suptitle('MSE 与 L1 损失的预测帧视觉对照')
figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'result_exp6_pred_compare.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(figure)